# Crypto Sweep Trading Signal System
Implementation of a range-sweep strategy with volume confirmation and trailing stop losses.

In [ ]:
# 🔹 CELL 1 — CONFIGURATION
import os
import sys

# Add project root to sys.path before importing config
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

import config

CONFIG = {
    "symbols": config.CRYPTO_TICKERS, # Pulls futures tickers (e.g. BTC/USDT:USDT)
    "timeframe": "1h",
    "exchange_id": config.EXCHANGE,   # Uses 'binanceusdm'
    
    # Strategy Parameters
    "range_lookback": 65,
    "vol_len": 21,
    "vol_multiplier": 1.5,
    "atr_len": 14,
    "atr_mult": 2.0,
    "min_range_perc": 3.0,
    "tp_mode": "Mid Range", # Options: "Mid Range" or "Full Range"
    
    # General Settings
    "fetch_limit": 1000,
    "polling_interval": 60  # Seconds
}

In [ ]:
# 🔹 CELL 2 — IMPORTS

import pandas as pd
import numpy as np
import ccxt
import requests
import time
import schedule

from utils.data_utils import update_ticker
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Initialize Exchange using System Config
exchange = getattr(ccxt, config.EXCHANGE)()

In [ ]:
# 🔹 CELL 3 — TELEGRAM FUNCTION

def send_telegram(message):
    """Sends a notification via Telegram Bot API."""
    token = config.TELEGRAM_BOT_TOKEN
    chat_id = config.TELEGRAM_CHAT_ID
    url = f"https://api.telegram.org/bot{token}/sendMessage"
    
    payload = {
        "chat_id": chat_id,
        "text": message,
        "parse_mode": "Markdown"
    }
    
    try:
        response = requests.post(url, json=payload, timeout=10)
        return response.json()
    except Exception as e:
        print(f"[!] Telegram Error: {e}")
        return None

In [ ]:
# 🔹 CELL 4 — DATA FETCHING

def fetch_and_sync_layer_a(symbol):
    """Uses Project Sentinel's Layer A logic to sync CSV base and return working DF."""
    try:
        # update_ticker uses filesystem paths safe for futures (e.g. BTC_USDT_USDT_1h.csv)
        working_df = update_ticker(symbol, asset_type='crypto')
        return working_df
    except Exception as e:
        print(f"[!] Snowball Sync Error for {symbol}: {e}")
        return None

In [ ]:
# 🔹 CELL 5 — INDICATORS

def calculate_atr(df, period=14):
    """Calculates Average True Range manually using pandas."""
    high_low = df['high'] - df['low']
    high_prev_close = (df['high'] - df['close'].shift(1)).abs()
    low_prev_close = (df['low'] - df['close'].shift(1)).abs()
    
    tr = pd.concat([high_low, high_prev_close, low_prev_close], axis=1).max(axis=1)
    atr = tr.rolling(window=period).mean()
    return atr

def apply_indicators(df):
    """Adds required indicator columns to the DataFrame."""
    # ATR
    df['atr'] = calculate_atr(df, CONFIG['atr_len'])
    
    # Range Levels (Rolling window)
    df['rangeHigh'] = df['high'].rolling(window=CONFIG['range_lookback']).max()
    df['rangeLow'] = df['low'].rolling(window=CONFIG['range_lookback']).min()
    
    # Volume MA
    df['vol_ma'] = df['volume'].rolling(window=CONFIG['vol_len']).mean()
    
    return df

In [ ]:
# 🔹 CELL 6 — STRATEGY LOGIC

def evaluate_strategy(df):
    """Applies strategy logic to find sweep signals."""
    # Range Logic
    df['rangeSizePerc'] = (df['rangeHigh'] - df['rangeLow']) / df['close'] * 100
    df['validRange'] = df['rangeSizePerc'] > CONFIG['min_range_perc']
    
    # Volume Spike
    df['volSpike'] = df['volume'] > (df['vol_ma'] * CONFIG['vol_multiplier'])
    
    # Sweeps (Shift rangeHigh/Low to get 'previous' values relative to current candle)
    df['prev_rangeHigh'] = df['rangeHigh'].shift(1)
    df['prev_rangeLow'] = df['rangeLow'].shift(1)
    
    # sweepHigh: high > previous rangeHigh AND close < current rangeHigh
    df['sweepHigh'] = (df['high'] > df['prev_rangeHigh']) & (df['close'] < df['rangeHigh'])
    
    # sweepLow: low < previous rangeLow AND close > current rangeLow
    df['sweepLow'] = (df['low'] < df['prev_rangeLow']) & (df['close'] > df['rangeLow'])
    
    # Entry Conditions
    df['longCondition'] = df['validRange'] & df['sweepLow'] & df['volSpike']
    df['shortCondition'] = df['validRange'] & df['sweepHigh'] & df['volSpike']
    
    # SL Base Calculations
    df['baseLongSL'] = df['close'] - (df['atr'] * CONFIG['atr_mult'])
    df['baseShortSL'] = df['close'] + (df['atr'] * CONFIG['atr_mult'])
    
    # TP Calculations
    df['midRange'] = (df['rangeHigh'] + df['rangeLow']) / 2
    if CONFIG['tp_mode'] == "Mid Range":
        df['longTP'] = df['midRange']
        df['shortTP'] = df['midRange']
    else:
        df['longTP'] = df['rangeHigh']
        df['shortTP'] = df['rangeLow']
        
    return df

In [ ]:
# 🔹 CELL 7 — SIGNAL GENERATION

# Track state for each symbol
signal_state = {
    symbol: {
        "current_pos": None, 
        "trailing_sl": 0.0, 
        "tp": 0.0,
        "last_processed_time": None
    } for symbol in CONFIG['symbols']
}

def process_signals(symbol, df):
    """Checks for new signals and updates state."""
    # We only look at the last closed candle (index -2)
    # index -1 is the currently forming candle
    last_closed = df.iloc[-2]
    candle_time = last_closed['timestamp']
    
    state = signal_state[symbol]
    
    # Prevent re-processing the same candle (fixes the 'repeating' issue)
    if state["last_processed_time"] == candle_time:
        return
    
    # CHECK FOR EXITS (TP/SL Hit)
    if state['current_pos'] == "LONG":
        if last_closed['low'] <= state['trailing_sl']:
            msg = f"🛑 *EXIT LONG {symbol}* (Stop Loss)\nPrice: {last_closed['close']:.4f}"
            send_telegram(msg)
            state['current_pos'] = None
        elif last_closed['high'] >= state['tp']:
            msg = f"🎯 *EXIT LONG {symbol}* (Take Profit)\nPrice: {last_closed['close']:.4f}"
            send_telegram(msg)
            state['current_pos'] = None
    elif state['current_pos'] == "SHORT":
        if last_closed['high'] >= state['trailing_sl']:
            msg = f"🛑 *EXIT SHORT {symbol}* (Stop Loss)\nPrice: {last_closed['close']:.4f}"
            send_telegram(msg)
            state['current_pos'] = None
        elif last_closed['low'] <= state['tp']:
            msg = f"🎯 *EXIT SHORT {symbol}* (Take Profit)\nPrice: {last_closed['close']:.4f}"
            send_telegram(msg)
            state['current_pos'] = None

    state["last_processed_time"] = candle_time
    
    # Check for NEW LONG
    if last_closed['longCondition'] and state['current_pos'] != "LONG":
        state['current_pos'] = "LONG"
        state['trailing_sl'] = last_closed['baseLongSL']
        state['tp'] = last_closed['longTP']
        
        msg = f"🚀 *LONG {symbol}*\nPrice: {last_closed['close']:.4f}\nSL: {state['trailing_sl']:.4f}\nTP: {state['tp']:.4f}"
        print(f"[{datetime.now()}] SIGNAL: {msg}")
        send_telegram(msg)
        
    # Check for NEW SHORT
    elif last_closed['shortCondition'] and state['current_pos'] != "SHORT":
        state['current_pos'] = "SHORT"
        state['trailing_sl'] = last_closed['baseShortSL']
        state['tp'] = last_closed['shortTP']
        
        msg = f"🔻 *SHORT {symbol}*\nPrice: {last_closed['close']:.4f}\nSL: {state['trailing_sl']:.4f}\nTP: {state['tp']:.4f}"
        print(f"[{datetime.now()}] SIGNAL: {msg}")
        send_telegram(msg)
        
    # Handle Trailing SL if in a position
    elif state['current_pos'] == "LONG":
        # Long: SL = max(previous SL, new baseLongSL)
        new_sl = max(state['trailing_sl'], last_closed['baseLongSL'])
        if new_sl > state['trailing_sl']:
            state['trailing_sl'] = new_sl
            print(f"[{symbol}] Trailing SL moved up to {new_sl:.4f}")
            
    elif state['current_pos'] == "SHORT":
        # Short: SL = min(previous SL, new baseShortSL)
        new_sl = min(state['trailing_sl'], last_closed['baseShortSL'])
        if new_sl < state['trailing_sl']:
            state['trailing_sl'] = new_sl
            print(f"[{symbol}] Trailing SL moved down to {new_sl:.4f}")

In [ ]:
# 🔹 CELL 8 — MAIN LOOP

def run_strategy_cycle():
    """Main job executed by the scheduler."""
    print(f"--- Starting Scan: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ---")
    for symbol in CONFIG['symbols']:
        try:
            # 1. Fetch & Sync with Local CSV Base (Layer A)
            df = fetch_and_sync_layer_a(symbol)
            
            if df is None or len(df) < CONFIG['range_lookback']:
                print(f"[!] {symbol}: Insufficient data.")
                continue
                
            # 2. Indicators
            df = apply_indicators(df)
            
            # 3. Strategy Logic
            df = evaluate_strategy(df)
            
            # 4. Process Signals & Check for SL/TP Exits
            process_signals(symbol, df)
            
        except Exception as e:
            print(f"[!] Loop error for {symbol}: {e}")

# Setup Scheduler
schedule.every(CONFIG['polling_interval']).seconds.do(run_strategy_cycle)

try:
    print('Scheduler running. Press Ctrl+C to stop.')
    
    # Run once immediately on start
    run_strategy_cycle()
    
    while True:
        schedule.run_pending()
        time.sleep(1)
except KeyboardInterrupt:
    print('Scheduler stopped.')